In [1]:
# imports

import sys
import argparse
import random
import re
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from rdkit import Chem
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

In [2]:
# tokenization for SMILES

TOKEN_PATTERN = re.compile(
    r"(\[[^\]]+\]|Br|Cl|Si|Na|Li|Mg|Ca|Al|@@?|=|#|-|\+|\\|/|\(|\)|\.|:|~|@|\?|>|\*|\$|%[0-9]{2}|[0-9]|[A-Za-z])"
)

def tokenize(smiles):
    """
    breaks down SMILES strings in useful tokens
    note to self: Br & Cl shouldnt be treated as 'B' + 'r'or 'C' + 'l'
    """
    tokens = TOKEN_PATTERN.findall(smiles)
    if "".join(tokens) != smiles:
        return None
    return tokens

def canonicalize(smiles):
    """
    checks if SMILES is valid and returns canonical SMILES - otherwise NONE
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return None

    return Chem.MolToSmiles(mol, canonical=True)

def randomized_smiles(smiles, n=5):
    """
    generates several alternative SMILES notations for the same molecule (should improve training)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    out = set()
    canonical = Chem.MolToSmiles(mol, canonical=True)
    out.add(canonical)

    for _ in range(n):
        try:
            s = Chem.MolToSmiles(mol, canonical=False, doRandom=True)
            out.add(s)
        except Exception:
            pass

    return list(out)

In [3]:
# dataset

class SmilesDataset(Dataset):
    def __init__(self, smiles_list, stoi, max_len):
        self.samples = []
        self.stoi = stoi
        self.max_len = max_len

        for smi in smiles_list:
            toks = tokenize(smi)
            if toks is None:
                continue

            toks = ["<bos>"] + toks + ["<eos>"]

            if len(toks) <= max_len:
                ids = [stoi[t] for t in toks]
                self.samples.append(ids)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]

        x = ids[:-1]
        y = ids[1:]

        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


def collate_batch(batch, pad_id):
    """
    padding function so SMILES of different lengths can coexist in the same batch
    """
    xs, ys = zip(*batch)

    max_len = max(len(x) for x in xs)

    x_pad = torch.full((len(xs), max_len), pad_id, dtype=torch.long)
    y_pad = torch.full((len(xs), max_len), pad_id, dtype=torch.long)

    for i, (x, y) in enumerate(zip(xs, ys)):
        x_pad[i, :len(x)] = x
        y_pad[i, :len(y)] = y

    return x_pad, y_pad

In [4]:
# GRU language model

class SmilesGRU(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, hidden_dim=512, num_layers=3, dropout=0.2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim)

        self.gru = nn.GRU(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        out, hidden = self.gru(emb, hidden)
        logits = self.output(out)
        return logits, hidden

In [5]:
# sampling

def sample_next_token(logits, temperature=0.9, top_k=20):
    """
    chooses next token probabilistically
    temperature small  => more conservative
    temperature big => more creative but also more invalid SMILES
    """
    logits = logits / temperature

    if top_k is not None and top_k > 0:
        values, indices = torch.topk(logits, k=min(top_k, logits.size(-1)))
        probs = torch.softmax(values, dim=-1)
        chosen = torch.multinomial(probs, num_samples=1)
        return indices[chosen].item()

    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()


@torch.no_grad()
def generate_one(model, stoi, itos, device, max_len=120, temperature=0.9, top_k=20):
    model.eval()

    bos_id = stoi["<bos>"]
    eos_id = stoi["<eos>"]

    x = torch.tensor([[bos_id]], dtype=torch.long, device=device)
    hidden = None

    tokens = []

    for _ in range(max_len):
        logits, hidden = model(x, hidden)
        next_logits = logits[0, -1]

        next_id = sample_next_token(next_logits, temperature=temperature, top_k=top_k)

        if next_id == eos_id:
            break

        token = itos[next_id]

        if token in {"<bos>", "<pad>"}:
            continue

        tokens.append(token)
        x = torch.tensor([[next_id]], dtype=torch.long, device=device)

    return "".join(tokens)

In [6]:
# training

def train_model(model, loader, pad_id, device, epochs=10, lr=1e-3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

    model.to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_batches = 0

        pbar = tqdm(loader, desc=f"Epoch {epoch}/{epochs}")

        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits, _ = model(x)

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1),
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            total_batches += 1

            pbar.set_postfix(loss=total_loss / total_batches)

        print(f"Epoch {epoch}: loss={total_loss / total_batches:.4f}")

In [7]:
# load smiles

def load_smiles(path):
    smiles = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                smiles.append(s)
    return smiles

In [8]:
# main 

def main():
    args = SimpleNamespace(
        train="smiles_train.txt",
        sample="sample_submission.txt",
        out="submission4.txt",
        num=10000, #upped form none
        epochs=5, # reduced from 12 and upped from 3
        batch_size=256, #upped from 128
        aug=0, # 4 is way to big
        max_len=120,
        temperature=0.90, #tried 0.85 and 0.75 and 0.7
        top_k=30, #tried 25 and 20 and 15
        seed=42,
    )

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # load SMILES
    train_smiles_raw = load_smiles(args.train)

    if args.num is None:
        if Path(args.sample).exists():
            args.num = len(load_smiles(args.sample))
        else:
            args.num = 10000

    print(f"Target number of molecules: {args.num}")

    # canoninacalize training molecules
    train_canon = set()
    clean_train = []

    for smi in tqdm(train_smiles_raw, desc="Canonicalizing train set"):
        can = canonicalize(smi)
        if can is not None:
            train_canon.add(can)
            clean_train.append(can)

    print(f"Valid training molecules: {len(clean_train)}")
    print(f"Unique canonical training molecules: {len(train_canon)}")

    # randomized SMILES for better training
    augmented = []

    for smi in tqdm(clean_train, desc="Augmenting SMILES"):
        augmented.extend(randomized_smiles(smi, n=args.aug))

    augmented = list(set(augmented))
    random.shuffle(augmented)

    print(f"Augmented training SMILES: {len(augmented)}")

    # build vocabular
    token_set = {"<pad>", "<bos>", "<eos>"}

    for smi in augmented:
        toks = tokenize(smi)
        if toks is not None:
            token_set.update(toks)

    itos = sorted(token_set)
    stoi = {tok: i for i, tok in enumerate(itos)}

    pad_id = stoi["<pad>"]

    print(f"Vocabulary size: {len(itos)}")
    print("Vocabulary:", itos)

    # create dataset, dataloader and model
    dataset = SmilesDataset(augmented, stoi, max_len=args.max_len)

    loader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=True,
        collate_fn=lambda batch: collate_batch(batch, pad_id),
        num_workers=0,
        pin_memory=(device == "cuda"),
    )
     
    model = SmilesGRU(
        vocab_size=len(itos),
        emb_dim=128,
        hidden_dim=256,
        num_layers=2,
        dropout=0.2,
    )
    
    train_model(model, loader, pad_id, device, epochs=args.epochs)

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "itos": itos,
            "stoi": stoi,
            "args": vars(args),
        },
        "smiles_gru_checkpoint_2.pt"
    )

    print("Saved model to smiles_gru_checkpoint3.pt")

    # generate and filter
    generated = []
    generated_set = set()

    attempts = 0
    max_attempts = args.num * 300

    pbar = tqdm(total=args.num, desc="Generating molecules")

    while len(generated) < args.num and attempts < max_attempts:
        attempts += 1

        smi = generate_one(
            model=model,
            stoi=stoi,
            itos=itos,
            device=device,
            max_len=args.max_len,
            temperature=args.temperature,
            top_k=args.top_k,
        )

        can = canonicalize(smi)

        if can is None:
            continue

        # Novelty: not in training
        if can in train_canon:
            continue

        # Uniqueness: not two times in submission
        if can in generated_set:
            continue

        # simple causality filter
        mol = Chem.MolFromSmiles(can)
        if mol is None:
            continue

        num_atoms = mol.GetNumAtoms()

        # aviod very small or large molecules
        if num_atoms < 5 or num_atoms > 80:
            continue

        generated.append(can)
        generated_set.add(can)
        pbar.update(1)

    pbar.close()

    if len(generated) < args.num:
        print(f"WARNING: only generated {len(generated)} molecules out of {args.num}")

    with open(args.out, "w", encoding="utf-8") as f:
        for smi in generated:
            f.write(smi + "\n")

    print(f"Wrote {len(generated)} molecules to {args.out}")
    print(f"Attempts: {attempts}")

    novelty = 100.0 * sum(s not in train_canon for s in generated) / max(1, len(generated))
    uniqueness = 100.0 * len(set(generated)) / max(1, len(generated))

    valid_count = sum(canonicalize(s) is not None for s in generated)
    validity = 100.0 * valid_count / max(1, len(generated))

    print(f"Estimated novelty:   {novelty:.2f}%")
    print(f"Estimated uniqueness:{uniqueness:.2f}%")
    print(f"Estimated validity:  {validity:.2f}%")

In [9]:
main()

Using device: cuda
Target number of molecules: 10000


Canonicalizing train set: 100%|██████████| 1272851/1272851 [11:11<00:00, 1894.22it/s]


Valid training molecules: 1272851
Unique canonical training molecules: 1272851


Augmenting SMILES: 100%|██████████| 1272851/1272851 [09:08<00:00, 2319.72it/s]


Augmented training SMILES: 1272851
Vocabulary size: 100
Vocabulary: ['#', '%10', '%11', '(', ')', '-', '1', '2', '3', '4', '5', '6', '7', '8', '9', '<bos>', '<eos>', '<pad>', '=', 'B', 'Br', 'C', 'Cl', 'F', 'I', 'N', 'O', 'P', 'S', '[B-]', '[BH-]', '[BH2-]', '[BH3-]', '[B]', '[Br+2]', '[Br-]', '[C+]', '[C-]', '[CH+]', '[CH-]', '[CH2+]', '[CH2]', '[CH]', '[Cl+2]', '[Cl+3]', '[Cl+]', '[Cl-]', '[F+]', '[F-]', '[I+2]', '[I+3]', '[I+]', '[IH2]', '[IH]', '[N+]', '[N-]', '[NH+]', '[NH-]', '[NH2+]', '[NH3+]', '[N]', '[O+]', '[O-]', '[OH+]', '[O]', '[P+]', '[P-]', '[PH2+]', '[PH]', '[S+]', '[S-]', '[SH]', '[Se+]', '[Se-]', '[SeH2]', '[SeH]', '[Se]', '[Si-]', '[SiH-]', '[SiH2]', '[SiH]', '[Si]', '[c+]', '[c-]', '[cH+]', '[cH-]', '[n+]', '[n-]', '[nH+]', '[nH]', '[o+]', '[s+]', '[se+]', '[se]', 'b', 'c', 'n', 'o', 'p', 's']


...

Epoch 4: loss=0.6558


Epoch 5/5: 100%|██████████| 4973/4973 [05:43<00:00, 14.47it/s, loss=0.649]


Epoch 5: loss=0.6490
Saved model to smiles_gru_checkpoint3.pt


Generating molecules: 100%|██████████| 10000/10000 [15:18<00:00, 10.89it/s]


Wrote 10000 molecules to submission4.txt
Attempts: 11611
Estimated novelty:   100.00%
Estimated uniqueness:100.00%
Estimated validity:  100.00%


In [12]:
import argparse
import os
import shutil
import sys
import tempfile
import time

submission_path = "submission4.txt"
target_path = "evaluation.tar"

print("Python:", sys.executable, flush=True)
print("Working directory:", os.getcwd(), flush=True)
print("Submission exists:", os.path.exists(submission_path), flush=True)
print("Target exists:", os.path.exists(target_path), flush=True)

with open(submission_path, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

print("Submission lines:", len(lines), flush=True)
print("First line:", lines[0] if lines else None, flush=True)

t0 = time.time()

with tempfile.TemporaryDirectory() as tmpdir:
    print("Unpacking evaluation.tar...", flush=True)
    shutil.unpack_archive(target_path, tmpdir)
    print(f"Unpacked after {time.time() - t0:.1f}s", flush=True)

    sys.path.insert(0, tmpdir)

    print("Importing get_metric...", flush=True)
    from evaluation.evaluate_submission import get_metric
    print(f"Imported after {time.time() - t0:.1f}s", flush=True)

    args = argparse.Namespace()
    args.submission = submission_path
    args.target = target_path
    args.trainset = os.path.join(tmpdir, "evaluation/data/smiles_train.txt")
    args.teststats = os.path.join(tmpdir, "evaluation/data/test_stats.p")

    print("Trainset exists:", os.path.exists(args.trainset), flush=True)
    print("Teststats exists:", os.path.exists(args.teststats), flush=True)

    print("Starting FCD calculation...", flush=True)
    metric_value = get_metric(args, "fcd")
    print(f"Finished after {time.time() - t0:.1f}s", flush=True)

    print("FCD:", metric_value)

Python: c:\Users\stec\anaconda3\envs\ails\python.exe
Working directory: c:\Users\stec\PycharmProjects\Master_AI\2nd_Semester\UE_AI_and_Life_Sciences\Generation_Challenge
Submission exists: True
Target exists: True
Submission lines: 10000
First line: CC(=O)NC(Cc1ccccc1)C(=O)N1CCN(S(=O)(=O)c2ccc3c(c2)OCCO3)CC1
Unpacking evaluation.tar...
Unpacked after 0.5s
Importing get_metric...
Imported after 1.2s
Trainset exists: True
Teststats exists: True
Starting FCD calculation...


c:\Users\stec\anaconda3\envs\ails\lib\site-packages\fcd\fcd.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_config = torch.load(model_path)


Finished after 26.7s
FCD: 0.9893442732530815
